## Comparison of experimental and calculated redox potentials for alternative protonation states

This analysis demonstrates how OFBO links an experimentally measured redox potential
to calculated redox potentials associated with different reaction representations of the
same molecule.

The selected case is benzo[g]phthalazine-5,10-dione, for which the mapped dataset
contains an experimental redox potential measured at pH 14 and three calculated redox
potentials corresponding to protonated, neutral, and deprotonated reaction forms.

Comparing these values supports interpretation of the molecular form most consistent
with the experimental alkaline condition. In this example, the calculated potential of
the deprotonated form can be compared directly with the pH 14 experimental potential.

The protonation-state labels are derived from the reaction identifiers defined in the
current OBDA mapping. A future extension could represent protonation states explicitly
using dedicated molecular-state or acid–base relations.

In [ ]:
from pathlib import Path
from io import StringIO

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import Markdown, display


ONTOP_ENDPOINT = "http://localhost:8080/sparql"
TIMEOUT_SECONDS = 120

## 1. SPARQL query and retrieval of experimental and calculated redox potentials

The query retrieves experimental and calculated redox potentials for molecules reported
in the study `10.1021/acs.chemmater.7b04220`.

Experimental potentials are retrieved for measurements performed at pH 14, together
with the molecular InChIKey, IUPAC name, potential reference, and the associated
experimental chemical system. The query additionally retrieves calculated redox
potentials for protonated, neutral, and deprotonated reaction representations of the
same molecule.

For the calculated reaction forms, the query also retrieves the SMILES representations
of the oxidized reactant and reduced product where available. These structural
representations make the protonation-state differences explicit and allow the calculated
reaction forms to be compared with the experimental alkaline-condition measurement.

The shared molecular InChIKey links the experimental observation to the corresponding
calculated reaction representations. This enables comparison of the measured pH 14
potential with calculated redox potentials for alternative protonation states, supporting
interpretation of the most plausible redox form under strongly basic conditions.

In [2]:
query_file = Path("queries/query_protonation_state_comparison.rq")
query = query_file.read_text(encoding="utf-8")

display(Markdown(f"```sparql\n{query}\n```"))

```sparql
PREFIX ofbo:   <https://w3id.org/ofbo#>
PREFIX schema: <http://schema.org/>
PREFIX sio:    <http://semanticscience.org/resource/>
PREFIX prov:   <http://www.w3.org/ns/prov#>
PREFIX qudt:   <http://qudt.org/schema/qudt/>
PREFIX rxno:   <http://purl.obolibrary.org/obo/RXNO_>

SELECT
    ?inchiKey
    ?iupacName
    ?reactionForm
    ?potential
    ?reference
    ?reactantSmiles
    ?productSmiles
WHERE {
    {
        # Experimental potential measured at pH 14.
        ?molecule schema:inChIKey ?inchiKey ;
                  schema:iupacName ?iupacName ;
                  ofbo:OFBO_0009206 ?reaction .

        ?potentialProperty a ofbo:OFBO_0001104 ;
                           sio:SIO_000011 ?reaction ;
                           qudt:numericValue ?potential ;
                           ofbo:OFBO_0009202 ?reference ;
                           prov:wasGeneratedBy ?experiment ;
                           prov:hadPrimarySource
                               <https://doi.org/10.1021/acs.chemmater.7b04220> .

        ?experiment a ofbo:OFBO_0008500 ;
                    prov:used ?system .

        ?system sio:SIO_000223 ?pHProperty .

        ?pHProperty a ofbo:OFBO_0001016 ;
                    qudt:numericValue 14 .

        BIND("Experimental, pH 14" AS ?reactionForm)
    }

    UNION

    {
        # Calculated protonated, neutral, and deprotonated reaction forms.
        ?molecule schema:inChIKey ?inchiKey ;
                  schema:iupacName ?iupacName ;
                  ofbo:OFBO_0009206 ?reaction .

        ?potentialProperty a ofbo:OFBO_0001104 ;
                           sio:SIO_000011 ?reaction ;
                           qudt:numericValue ?potential ;
                           ofbo:OFBO_0009202 ?reference ;
                           prov:wasGeneratedBy ?calculation ;
                           prov:hadPrimarySource
                               <https://doi.org/10.1021/acs.chemmater.7b04220> .

        ?calculation a ofbo:OFBO_0008001 .

        OPTIONAL {
            ?reaction rxno:0000424 ?reactant .
            ?reactant schema:smiles ?reactantSmiles .
        }

        OPTIONAL {
            ?reaction rxno:0000423 ?product .
            ?product schema:smiles ?productSmiles .
        }

        BIND(
            IF(
                CONTAINS(STR(?reaction), "_DFT_PROT"),
                "Calculated, protonated",
                IF(
                    CONTAINS(STR(?reaction), "_DFT_NEUT"),
                    "Calculated, neutral",
                    IF(
                        CONTAINS(STR(?reaction), "_DFT_DEPROT"),
                        "Calculated, deprotonated",
                        "Other calculated form"
                    )
                )
            )
            AS ?reactionForm
        )

        FILTER(
            CONTAINS(STR(?reaction), "_DFT_PROT")
            || CONTAINS(STR(?reaction), "_DFT_NEUT")
            || CONTAINS(STR(?reaction), "_DFT_DEPROT")
        )
    }
}
ORDER BY ?inchiKey ?reactionForm
```

In [3]:
response = requests.post(
    ONTOP_ENDPOINT,
    data={"query": query},
    headers={"Accept": "text/csv"},
    timeout=TIMEOUT_SECONDS,
)

response.raise_for_status()

protonation_results = pd.read_csv(
    StringIO(response.text)
).drop_duplicates()

protonation_results["potential"] = pd.to_numeric(
    protonation_results["potential"],
    errors="coerce",
)

protonation_results = protonation_results.dropna(
    subset=["inchiKey", "reactionForm", "potential"]
).copy()

print(
    f"Retrieved {len(protonation_results)} "
    "experimental or calculated potential records."
)

print(
    f"Retrieved {protonation_results['inchiKey'].nunique()} "
    "unique molecules."
)

protonation_results[
    [
        "inchiKey",
        "iupacName",
        "reactionForm",
        "potential",
        "reference",
        "reactantSmiles",
        "productSmiles",
    ]
]

Retrieved 3 experimental or calculated potential records.
Retrieved 1 unique molecules.


,inchiKey,iupacName,reactionForm,potential,reference,reactantSmiles,productSmiles
0,VUCRHUZTFILZPG-UHFFFAOYSA-N,"benzo[g]phthalazine-5,10-dione","Calculated, deprotonated",-0.448,SHE,O=C1c2ccccc2C(=O)c2cnncc21,[O-]c2c1ccccc1c([O-])c3cnncc23
1,VUCRHUZTFILZPG-UHFFFAOYSA-N,"benzo[g]phthalazine-5,10-dione","Calculated, neutral",0.328,SHE,O=C1c2ccccc2C(=O)c2cnncc21,Oc2c1ccccc1c(O)c3cnncc23
2,VUCRHUZTFILZPG-UHFFFAOYSA-N,"benzo[g]phthalazine-5,10-dione","Calculated, protonated",0.590,SHE,O=c2c1ccccc1c(=O)c3c[nH+][nH+]cc23,Oc2c1ccccc1c(O)c3c[nH+][nH+]cc23


## 2. Interpretation of calculated protonation-state-dependent redox potentials

The query retrieves three calculated redox potentials for benzo[g]phthalazine-5,10-dione, each associated with an explicitly represented oxidized reactant and reduced product structure.

The calculated potential depends strongly on the protonation state:

- protonated form: 0.590 V versus SHE;
- neutral form: 0.328 V versus SHE; and
- deprotonated form: −0.448 V versus SHE.

The deprotonated reaction form therefore has a substantially more negative calculated redox potential than the neutral and protonated forms. Its reactant and product structures contain deprotonated oxygen atoms, represented by `[O-]`, whereas the neutral and protonated forms retain hydroxyl or protonated nitrogen groups.

This result is chemically consistent with the expectation that benzo[g]phthalazine-5,10-dione is predominantly deprotonated under strongly alkaline conditions such as pH 14. The ontology-based representation makes this comparison possible because each calculated potential is linked to a distinct reaction instance and to the corresponding reactant and product SMILES structures.

The displayed query result contains only the three calculated reaction forms. The experimental pH 14 potential is not present in this output and should be retrieved separately or included in the query before making a direct quantitative comparison between experiment and calculation.